# Airbnb Brussels: Data Cleaning

**Fatima Zahra Ben Khaled** · SYNTRA Data Analyst Eindproef


In [1]:
import pandas as pd

## Load both snapshots

In [2]:
june = pd.read_csv('../01_Raw_Data/listings June Brussels.csv', low_memory=False)
june['snapshot_date'] = '2025-06-21'

sept = pd.read_csv('../01_Raw_Data/listings September Brussels.csv', low_memory=False)
sept['snapshot_date'] = '2025-09-23'

print('June:     ', len(june), 'rows,', len(june.columns), 'columns')
print('September:', len(sept), 'rows,', len(sept.columns), 'columns')

June:      6721 rows, 80 columns
September: 6131 rows, 19 columns


In [3]:
june.head(2)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,snapshot_date
0,45026,https://www.airbnb.com/rooms/45026,20250621143231,2025-06-21,city scrape,Apartment raised basement,"Beautiful apartment, freshly repainted, 65 m2,...",NaN,https://a0.muscache.com/pictures/5622354/dd7be...,198732,...,4.88,4.73,NaN,f,1,1,0,0,0.9,2025-06-21
1,48180,https://www.airbnb.com/rooms/48180,20250621143231,2025-06-21,city scrape,in renewal,"Top Apart, perfect location 0 default, 2rooms ...",NaN,https://a0.muscache.com/pictures/miso/Hosting-...,219560,...,NaN,NaN,NaN,f,1,1,0,0,NaN,2025-06-21


In [4]:
# In June it's called neighbourhood_cleansed, in September it's just neighbourhood.
june = june.drop(columns=['neighbourhood'])
june = june.rename(columns={'neighbourhood_cleansed': 'neighbourhood'})

## Union the two snapshots


In [5]:
combined = pd.concat([june, sept], ignore_index=True)
print('Combined:', len(combined), 'rows')

Combined: 12852 rows


## Clean the price column

In [6]:
combined['price'] = combined['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
combined['price'] = pd.to_numeric(combined['price'], errors='coerce')

combined['price'].describe()

count    10766.000000
mean       116.454022
std        234.033151
min          8.000000
25%         67.000000
50%         89.000000
75%        124.000000
max       9999.000000
Name: price, dtype: float64

## Filter (drop missing or zero prices)

In [7]:
before = len(combined)
combined = combined[combined['price'] > 0]
print('Dropped', before - len(combined), 'rows. Kept', len(combined))

Dropped 2086 rows. Kept 10766


## Drop (keep only the 9 columns I need)

In [9]:
keep = ['id', 'host_id', 'neighbourhood', 'room_type',
        'latitude', 'longitude', 'price', 'availability_365',
        'snapshot_date']

combined = combined[keep]
combined.head(3)

,id,host_id,neighbourhood,room_type,latitude,longitude,price,availability_365,snapshot_date
0,45026,198732,Saint-Gilles,Entire home/apt,50.82677,4.34987,64.0,328,2025-06-21
1,48180,219560,Woluwe-Saint-Pierre,Entire home/apt,50.83771,4.40707,200.0,364,2025-06-21
2,52796,244722,Ixelles,Entire home/apt,50.83362,4.36057,83.0,222,2025-06-21


## Median (no row deletion, but I commit to using median in the dashboard)

In [10]:
print('Mean:  ', round(combined['price'].mean(), 2))
print('Median:', combined['price'].median())
print('Max:   ', combined['price'].max())

Mean:   116.45
Median: 89.0
Max:    9999.0


## Rename columns to match my star schema

In [12]:
combined = combined.rename(columns={
    'id':                'ListingID',
    'host_id':           'HostID',
    'neighbourhood':     'NeighbourhoodName',
    'room_type':         'RoomTypeName',
    'latitude':          'Latitude',
    'longitude':         'Longitude',
    'price':             'Price',
    'availability_365':  'Availability365',
    'snapshot_date':     'SnapshotDate'
})

combined.head(3)

,ListingID,HostID,NeighbourhoodName,RoomTypeName,Latitude,Longitude,Price,Availability365,SnapshotDate
0,45026,198732,Saint-Gilles,Entire home/apt,50.82677,4.34987,64.0,328,2025-06-21
1,48180,219560,Woluwe-Saint-Pierre,Entire home/apt,50.83771,4.40707,200.0,364,2025-06-21
2,52796,244722,Ixelles,Entire home/apt,50.83362,4.36057,83.0,222,2025-06-21


## Save the clean data

In [14]:
combined.to_csv('airbnb_brussels_2snapshots_clean.csv', index=False)
print('Saved', len(combined), 'rows to airbnb_brussels_2snapshots_clean.csv')

Saved 10766 rows to airbnb_brussels_2snapshots_clean.csv
